# 🤖 노인 가사 로봇 도우미 — 통합 Live Demo

## 시스템 개요

| Part | 기술 | 역할 |
|------|------|------|
| **Part 1 (AI)** | `MultiTaskResNet` | 손 제스처 분류 + 라인 트레이싱 |
| **Part 2 (Rule)** | Color Tracking (HSV) | Zone 도달 여부 판별 |
| **Part 3 (통합)** | State Machine | 세 상태를 연결하여 자율 동작 |

## 동작 흐름

```
  [WAITING]
     │  손 제스처 분류
     │  Class 1 (ㅗ 모양) or Class 2 (V자) or Class 3 (손가락 3)
     │
     ▼
  [LINE_TRACING]
     │  AI로 라인 추종 + 매 프레임 색상 Zone 체크
     │  색상 물체 면적 > 임계치
     ▼
  [FETCHED]
     │  로봇 정지 후 3초 대기
     ▼
  [WAITING] ← 복귀
```

## 사전 준비
- `best_model_multi.pth` 모델 파일이 같은 디렉터리에 있어야 합니다.
- 없으면 `Train_Model_Multi.ipynb` 실행 후 진행하세요.
- 각 Zone 색상(Yellow 기본)을 물건 위치에 배치하세요.

---
## Cell 1. 라이브러리 임포트 및 AI 모델 로드

`MultiTaskResNet` 은 ResNet18 백본을 공유하며 두 종류 헤드를 갖습니다:
- `xy_head` : 라인 트레이싱 (x, y 회귀 출력)
- `finger_head` : 손 제스처 분류 (클래스 0~5)

In [59]:
import time
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
import cv2
import numpy as np
import traitlets
import ipywidgets.widgets as widgets
from IPython.display import display
from jetbot import Robot, Camera, bgr8_to_jpeg
import torch.nn.functional as F

# ── MultiTaskResNet 정의 ──────────────────────────────────────────────────────
class MultiTaskResNet(nn.Module):
    """
    ResNet18 백본 공유 + 두 개의 Task Head
      mode='line'   → xy_head    (라인 트레이싱 x,y 회귀)
      mode='finger' → finger_head (손 제스처 클래스 0~5 분류)
    """
    def __init__(self):
        super(MultiTaskResNet, self).__init__()
        resnet = models.resnet18(pretrained=False)
        self.backbone    = nn.Sequential(*list(resnet.children())[:-1])
        self.xy_head     = nn.Linear(512, 2)
        self.finger_head = nn.Linear(512, 6)

    def forward(self, x, mode='line'):
        x = self.backbone(x)
        x = torch.flatten(x, 1)
        if mode == 'line':
            return self.xy_head(x)
        elif mode == 'finger':
            return self.finger_head(x)

# ── 디바이스 설정 및 모델 로드 ────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[INFO] Using device: {device}')

model = MultiTaskResNet()
try:
    model.load_state_dict(torch.load('best_model_multi.pth'))
    print('[OK] best_model_multi.pth 모델 로드 성공!')
except FileNotFoundError:
    print('[WARN] best_model_multi.pth 없음 — 랜덤 가중치 사용. 먼저 Train_Model_Multi.ipynb 실행!')
except Exception as e:
    print(f'[WARN] 모델 로드 실패: {e}')

model = model.to(device)
model.eval()
print('[OK] 모델 준비 완료.')

[INFO] Using device: cuda
[OK] best_model_multi.pth 모델 로드 성공!
[OK] 모델 준비 완료.


---
## Cell 2. 서보(TTLServo) 초기화

카메라 팬틸트 서보를 초기 위치(정면)로 이동합니다.
서보가 없거나 포트 오류 시 경고만 출력하고 계속 진행합니다.

In [60]:
# ── TTLServo 초기화 ───────────────────────────────────────────────────────────
servoCtrlTime = 0.001

try:
    from SCSCtrl import TTLServo
    ttl_available = True

    # 카메라 팬틸트 정면 초기화
    TTLServo.servoAngleCtrl(1, 0, 1, 150)   # 수평(Pan) : 0도(정면)
    time.sleep(servoCtrlTime)
    TTLServo.servoAngleCtrl(5, 0, 1, 150)   # 수직(Tilt): 0도(정면)
    time.sleep(servoCtrlTime)
    print('[OK] TTLServo 초기화 완료 (Pan=0°, Tilt=0°)')

    # ── 카메라 서보 헬퍼 함수 ────────────────────────────────────────────────
    def cameraUp(speed):    TTLServo.servoAngleCtrl(5,  25, 1, speed); time.sleep(servoCtrlTime)
    def cameraDown(speed):  TTLServo.servoAngleCtrl(5, -70, 1, speed); time.sleep(servoCtrlTime)
    def ptRight(speed):     TTLServo.servoAngleCtrl(1,  80, 1, speed); time.sleep(servoCtrlTime)
    def ptLeft(speed):      TTLServo.servoAngleCtrl(1, -80, 1, speed); time.sleep(servoCtrlTime)
    def tiltStop():         TTLServo.servoStop(5);                      time.sleep(servoCtrlTime)
    def panStop():          TTLServo.servoStop(1);                      time.sleep(servoCtrlTime)
    def servoCenter():
        TTLServo.servoAngleCtrl(1, 0, 1, 150); time.sleep(servoCtrlTime)
        TTLServo.servoAngleCtrl(5, 0, 1, 150); time.sleep(servoCtrlTime)

    # ── 로봇 팔 위치 함수 ────────────────────────────────────────────────────
    def init_arm():  # 분류 모드(WAITING) 용 팔 위치
        TTLServo.servoAngleCtrl(1,  0, 1, 150)
        TTLServo.servoAngleCtrl(2, 50, 1, 150)
        TTLServo.servoAngleCtrl(3, 0, 1, 150)
        TTLServo.servoAngleCtrl(4,  0, 1, 150)
        TTLServo.servoAngleCtrl(5, 50, 1, 150)
        time.sleep(1)
        print('[OK] 팔 → 분류 모드 위치 (init_arm)')

    def down_arm():  # 주행 모드(LINE_TRACING) 용 팔 위치
        TTLServo.servoAngleCtrl(5, -45, 1, 150)
        time.sleep(1)
        print('[OK] 팔 → 주행 모드 위치 (down_arm)')

    def arm_gotit():
        print("\r[동작] got it!", end='')
        GOT_IT_SPEED = 300
        GOT_IT_DELAY = 0.5
        init_arm()
        TTLServo.servoAngleCtrl(3, 20, 1, GOT_IT_SPEED)
        time.sleep(GOT_IT_DELAY)
        TTLServo.servoAngleCtrl(3, -20, 1, GOT_IT_SPEED)
        time.sleep(GOT_IT_DELAY)
        TTLServo.servoAngleCtrl(3, 20, 1, GOT_IT_SPEED)
        time.sleep(GOT_IT_DELAY)
        TTLServo.servoAngleCtrl(3, 0, 1, GOT_IT_SPEED)

    def arm_give():
        print("\r[동작] give!", end='')
        GIVE_SPEED = 150
        CIRCLE_SPEED = 250
        CIRCLE_DELAY = 0.15
        CIRCLE_COUNT = 2
        circle_points = [
            (0, -5), (14, -11), (20, -25), (14, -39),
            (0, -50), (-14, -39), (-20, -25), (-14, -11)
        ]
        TTLServo.servoAngleCtrl(1, 0, 1, GIVE_SPEED)
        TTLServo.servoAngleCtrl(2, -45, 1, GIVE_SPEED)
        TTLServo.servoAngleCtrl(3, -90, 1, GIVE_SPEED)
        TTLServo.servoAngleCtrl(4, 0, 1, GIVE_SPEED)
        TTLServo.servoAngleCtrl(5, 50, 1, GIVE_SPEED)
        time.sleep(5)
        TTLServo.servoAngleCtrl(4, -60, 1, GIVE_SPEED)
        time.sleep(1.5)
        TTLServo.servoAngleCtrl(2, 0, 1, GIVE_SPEED)
        time.sleep(2)
        for _ in range(CIRCLE_COUNT):
            for point_x, point_y in circle_points:
                TTLServo.servoAngleCtrl(1, point_x, 1, CIRCLE_SPEED)
                TTLServo.servoAngleCtrl(2, point_y, 1, CIRCLE_SPEED)
                time.sleep(CIRCLE_DELAY)
        TTLServo.servoAngleCtrl(1, 0, 1, GIVE_SPEED)
        TTLServo.servoAngleCtrl(2, 0, 1, GIVE_SPEED)
        TTLServo.servoAngleCtrl(3, -90, 1, GIVE_SPEED)
        TTLServo.servoAngleCtrl(5, 50, 1, GIVE_SPEED)

    init_arm()  # 시작 시 분류 모드 위치로 초기화

except Exception as e:
    ttl_available = False
    print(f'[WARN] TTLServo 사용 불가: {e}')
    def cameraUp(speed): pass
    def cameraDown(speed): pass
    def ptRight(speed): pass
    def ptLeft(speed): pass
    def tiltStop(): pass
    def panStop(): pass
    def servoCenter(): pass
    def init_arm(): pass
    def down_arm(): pass
    def arm_gotit(): pass
    def arm_give(): pass


[OK] TTLServo 초기화 완료 (Pan=0°, Tilt=0°)
[OK] 팔 → 분류 모드 위치 (init_arm)


---
## Cell 3. 카메라 & 로봇 초기화

In [61]:
camera = Camera.instance(width=224, height=224)
robot  = Robot()
print('[OK] 카메라 & 로봇 초기화 완료.')

[OK] 카메라 & 로봇 초기화 완료.


---
## Cell 4. 파라미터 설정

아래 값을 환경에 맞게 조정하세요.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# [Part 2] 색상 Zone 설정 (HSV)
# 목표 물건 색상에 맞게 수정하세요.
# ═══════════════════════════════════════════════════════════════════════════════

COLORS = {
    'YELLOW': {
        'LOWER': np.array([24, 100, 100]),
        'UPPER': np.array([44, 255, 255])
    },
    'RED': {
        'LOWER': np.array([160, 100, 100]),
        'UPPER': np.array([180, 255, 255])
    },
    'PURPLE': {
        'LOWER': np.array([125, 50, 50]),
        'UPPER': np.array([150, 255, 255])
    }
}

# 이 면적(픽셀²) 이상이면 물건에 도달한 것으로 판정
ZONE_AREA_THRESHOLD = 2500   # 환경에 따라 조정

# ═══════════════════════════════════════════════════════════════════════════════
# [Part 1] 라인 트레이싱 파라미터
# ═══════════════════════════════════════════════════════════════════════════════
LINE_SPEED        = 0.4    # 기본 전진 속도 (0.0 ~ 1.0)
LINE_STEERING_GAIN = 0.25  # PID 비례 계수 (값이 크면 더 민감하게 반응)

# ═══════════════════════════════════════════════════════════════════════════════
# [Part 3] State Machine 파라미터
# ═══════════════════════════════════════════════════════════════════════════════
CONFIDENCE_THRESHOLD = 0.80    # 제스처 인식 신뢰도 임계치 (0.0 ~ 1.0)
DANCE_DEBOUNCE_SEC   = 3.0     # 행동 후 재실행 방지 시간(초)
FETCHED_WAIT_SEC     = 3.0     # 물건 도달 후 WAITING 복귀까지 대기 시간(초)

# ── 제스처 클래스 매핑 ────────────────────────────────────────────────────────
#   학습 데이터의 클래스 레이블과 반드시 일치해야 합니다.
GESTURE_LABELS = {
    0: '주먹 (대기)',
    1: 'ㅗ 모양 (화난 모습)',
    2: 'V자 → 노란색 목적지 + 전달',
    3: '손가락 3 → 붉은색 목적지 + 전달',
    4: '손가락 4 (대기)',
    5: '손 전체 (대기)',
}

print('[OK] 파라미터 설정 완료.')
print(f'  Zone 색상 임계치 : {ZONE_AREA_THRESHOLD} px²')
print(f'  라인 속도        : {LINE_SPEED}')
print(f'  제스처 신뢰도    : {CONFIDENCE_THRESHOLD}')


[OK] 파라미터 설정 완료.
  Zone 색상 임계치 : 5000 px²
  라인 속도        : 0.75
  제스처 신뢰도    : 0.8


---
## Cell 5. 핵심 함수 정의

### 5-1. 이미지 전처리
### 5-2. 색상 Zone 체크 (Part 2)
### 5-3. 제스처 행동 함수 (Part 3)

In [63]:
import threading

# ── 5-1. 이미지 전처리 ────────────────────────────────────────────────────────
def preprocess(camera_value):
    """BGR numpy 이미지 → 모델 입력 텐서 (1, 3, 224, 224)"""
    x = cv2.cvtColor(camera_value, cv2.COLOR_BGR2RGB)
    x = x.transpose((2, 0, 1)).astype(np.float32) / 255.0
    x = torch.from_numpy(x)
    x = transforms.functional.normalize(
            x, [0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    return x.to(device).unsqueeze(0)


# ── 5-2. Part 2: 색상 Zone 체크 ──────────────────────────────────────────────
def check_zone(image_bgr, target_color):
    """
    HSV 색상 마스크로 목표 물건까지의 거리(면적)를 추정합니다.
    """
    hsv  = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
    if not target_color:
        return False, image_bgr.copy(), 0.0
        
    lower = COLORS[target_color]['LOWER']
    upper = COLORS[target_color]['UPPER']
    
    mask = cv2.inRange(hsv, lower, upper)
    mask = cv2.erode(mask,  None, iterations=2)
    mask = cv2.dilate(mask, None, iterations=2)

    cnts = cv2.findContours(mask.copy(),
                            cv2.RETR_EXTERNAL,
                            cv2.CHAIN_APPROX_SIMPLE)[-2]

    annotated = image_bgr.copy()
    reached   = False
    area      = 0.0

    if len(cnts) > 0:
        c    = max(cnts, key=cv2.contourArea)
        area = cv2.contourArea(c)
        ((bx, by), radius) = cv2.minEnclosingCircle(c)

        color = (0, 255, 0) if area > ZONE_AREA_THRESHOLD else (0, 255, 255)
        cv2.rectangle(annotated,
                      (int(bx - radius), int(by + radius)),
                      (int(bx + radius), int(by - radius)),
                      color, 2)

        if area > ZONE_AREA_THRESHOLD:
            reached = True
            cv2.putText(annotated, f'✓ REACHED  area={int(area)}',
                        (5, 20), cv2.FONT_HERSHEY_SIMPLEX,
                        0.5, (0, 255, 0), 2, cv2.LINE_AA)
        else:
            cv2.putText(annotated, f'Tracking  area={int(area)}',
                        (5, 20), cv2.FONT_HERSHEY_SIMPLEX,
                        0.5, (255, 255, 0), 1, cv2.LINE_AA)
    else:
        cv2.putText(annotated, 'Searching color zone...',
                    (5, 20), cv2.FONT_HERSHEY_SIMPLEX,
                    0.5, (180, 180, 180), 1, cv2.LINE_AA)

    return reached, annotated, area


# ── 5-3. Part 3: 제스처 행동 함수 ────────────────────────────────────────────
is_acting = False
def run_macro(action_func, callback=None):
    global is_acting
    if is_acting: return
    is_acting = True
    def run():
        action_func()
        global is_acting
        is_acting = False
        if callback: callback()
    threading.Thread(target=run).start()

def perform_dance():
    """
    'ㅗ' 제스처 인식 시 실행:
    좌 → 우 → 좌 → 우 ('도리도리') 후 정지 (화난 모습)
    """
    print('[ACTION] Dance: ← → ← →')
    for _ in range(2):
        robot.left(0.45);  time.sleep(0.35)
        robot.right(0.45); time.sleep(0.35)
    robot.stop()
    print('[ACTION] Dance 완료.')


print('[OK] 핵심 함수 정의 완료.')


[OK] 핵심 함수 정의 완료.


---
## Cell 6. UI 위젯 생성

실시간 카메라 영상과 상태 정보를 표시합니다.

In [64]:
# ── 카메라 출력 위젯 ──────────────────────────────────────────────────────────
image_widget = widgets.Image(format='jpeg', width=224, height=224)

# ── 상태 표시 위젯 ────────────────────────────────────────────────────────────
state_label   = widgets.HTML(value='<h3 style="color:#4fc3f7">● State: WAITING 🖐️</h3>')
gesture_label = widgets.HTML(value='<p style="color:#aaa">제스처를 기다리는 중...</p>')
detail_label  = widgets.HTML(value='')
zone_label    = widgets.HTML(value='')

# ── 제스처 확률 바 (클래스 0~5) ──────────────────────────────────────────────
LABEL_NAMES = ['0:주먹', '1:ㅗ', '2:V자', '3:셋', '4:넷', '5:전체']
prob_bars = [
    widgets.FloatProgress(
        value=0.0, min=0.0, max=1.0,
        description=name,
        style={'description_width': '60px'},
        layout=widgets.Layout(width='300px')
    )
    for name in LABEL_NAMES
]

prob_box = widgets.VBox(
    [widgets.HTML('<b>손 제스처 확률</b>')] + prob_bars,
    layout=widgets.Layout(padding='10px')
)

info_box = widgets.VBox([
    state_label,
    gesture_label,
    detail_label,
    zone_label,
])

print('[OK] UI 위젯 생성 완료. (Cell 8에서 display 됩니다)')

[OK] UI 위젯 생성 완료. (Cell 8에서 display 됩니다)


---
## Cell 7. State Machine 메인 콜백 정의

카메라 프레임이 갱신될 때마다 `update()` 가 자동 호출됩니다.

### 상태 전이 규칙

| 현재 상태 | 조건 | 다음 상태 |
|-----------|------|-----------|
| WAITING | Class 1 (ㅗ) + conf ≥ threshold | WAITING (dance 후 복귀) |
| WAITING | Class 2 or 3 + conf ≥ threshold | LINE_TRACING |
| LINE_TRACING | Zone 면적 ≥ 임계치 | FETCHED |
| FETCHED | 3초 경과 | WAITING |

In [65]:
# ── 상태 변수 ─────────────────────────────────────────────────────────────────
class RobotState:
    WAITING      = 'WAITING'
    LINE_TRACING = 'LINE_TRACING'
    MACRO_RUNNING = 'MACRO_RUNNING'
    FETCHED      = 'FETCHED'

current_state   = RobotState.WAITING
fetched_at      = None
last_dance_time = 0.0
target_color    = None
current_mission = None  # 'TO_TARGET' or 'RETURN_HOME'

def update(change):
    global current_state, fetched_at, last_dance_time, target_color, current_mission

    frame = change['new']       # BGR numpy array (224×224)
    x_in  = preprocess(frame)   # 모델 입력 텐서

    # ══════════════════════════════════════════════════════════════════════════
    # WAITING — 손 제스처 인식
    # ══════════════════════════════════════════════════════════════════════════
    if current_state == RobotState.WAITING:
        robot.stop()
        image_widget.value = bgr8_to_jpeg(frame)
        state_label.value  = '<h3 style="color:#4fc3f7">● State: WAITING 🖐️</h3>'

        # 댄스 직후 쿨다운
        if time.time() - last_dance_time < DANCE_DEBOUNCE_SEC:
            remaining = DANCE_DEBOUNCE_SEC - (time.time() - last_dance_time)
            gesture_label.value = f'<p>⏳ 쿨다운 중... {remaining:.1f}s</p>'
            return

        # Part 1: 손 제스처 추론
        with torch.no_grad():
            logits = model(x_in, mode='finger')
            probs  = F.softmax(logits, dim=1).cpu().numpy().flatten()

        cls  = int(np.argmax(probs))
        conf = float(probs[cls])

        # 확률 바 업데이트
        for i in range(6):
            prob_bars[i].value = float(probs[i])

        gesture_label.value = (
            f'<p>👁 인식: <b>{GESTURE_LABELS.get(cls, str(cls))}</b>'
            f' — 신뢰도: <b>{conf:.2f}</b></p>'
        )

        # ── 행동 분기 ──────────────────────────────────────────────────────
        if conf >= CONFIDENCE_THRESHOLD:
            if cls == 1:
                # 1번 클래스: 흔들흔들(화난 모습) 후 WAITING 복귀
                print('[STATE] WAITING -> MACRO_RUNNING (Dance)')
                detail_label.value = '<p>💃 화난 모습 (Dance)!</p>'
                current_state = RobotState.MACRO_RUNNING
                def on_dance_done():
                    global current_state, last_dance_time
                    last_dance_time = time.time()
                    current_state = RobotState.WAITING
                run_macro(perform_dance, on_dance_done)

            elif cls in (2, 3):
                # 2번(Yellow), 3번(Red) 클래스: gotit -> line_tracing(to target)
                print(f'[STATE] WAITING -> MACRO_RUNNING (gotit, class={cls})')
                detail_label.value = '<p>🤖 Got It 모션 수행 중...</p>'
                
                target_color = 'YELLOW' if cls == 2 else 'RED'
                current_mission = 'TO_TARGET'
                
                current_state = RobotState.MACRO_RUNNING
                def on_gotit_done():
                    global current_state
                    current_state = RobotState.LINE_TRACING
                    down_arm()
                run_macro(arm_gotit, on_gotit_done)

    # ══════════════════════════════════════════════════════════════════════════
    # MACRO_RUNNING — 매크로 동작 수행 중
    # ══════════════════════════════════════════════════════════════════════════
    elif current_state == RobotState.MACRO_RUNNING:
        state_label.value = '<h3 style="color:#ffb74d">● State: MACRO_RUNNING ⚙️</h3>'
        image_widget.value = bgr8_to_jpeg(frame)
        # 매크로가 로봇을 제어할 수 있으므로 여기서 robot.stop()을 호출하지 않음

    # ══════════════════════════════════════════════════════════════════════════
    # LINE_TRACING — 라인 주행 + Zone 체크
    # ══════════════════════════════════════════════════════════════════════════
    elif current_state == RobotState.LINE_TRACING:
        state_label.value = '<h3 style="color:#81c784">● State: LINE_TRACING 🚗</h3>'

        # Part 2: Zone 체크 (색상 인식)
        reached, annotated, area = check_zone(frame, target_color)
        image_widget.value = bgr8_to_jpeg(annotated)
        zone_label.value   = f'<p>🎯 {target_color} Zone 면적: {int(area)} px² (임계: {ZONE_AREA_THRESHOLD})</p>'

        if reached:
            robot.stop()
            if current_mission == 'TO_TARGET':
                print('[STATE] LINE_TRACING -> MACRO_RUNNING (give)')
                detail_label.value = '<p>🎁 물건 전달 모션 수행 중...</p>'
                current_state = RobotState.MACRO_RUNNING
                current_mission = 'RETURN_HOME'
                target_color = 'PURPLE'
                def on_give_done():
                    global current_state
                    current_state = RobotState.LINE_TRACING
                    down_arm()
                run_macro(arm_give, on_give_done)
            elif current_mission == 'RETURN_HOME':
                print('[STATE] LINE_TRACING -> FETCHED')
                current_state = RobotState.FETCHED
                fetched_at = time.time()
            return

        # Part 1: 라인 트레이싱 추론
        with torch.no_grad():
            out = model(x_in, mode='line')
            xy  = out.detach().cpu().numpy().flatten()

        x_val, y_val = float(xy[0]), float(xy[1])
        angle = np.arctan2(x_val, y_val)
        pid   = angle * LINE_STEERING_GAIN

        left_speed  = max(min(LINE_SPEED + pid, 1.0), 0.0)
        right_speed = max(min(LINE_SPEED - pid, 1.0), 0.0)

        robot.left_motor.value  = left_speed
        robot.right_motor.value = right_speed

        detail_label.value = (
            f'<p>📐 x={x_val:.3f}  y={y_val:.3f} | '
            f'L={left_speed:.2f}  R={right_speed:.2f}</p>'
        )

    # ══════════════════════════════════════════════════════════════════════════
    # FETCHED — 도달 확인 → 3초 후 WAITING 복귀
    # ══════════════════════════════════════════════════════════════════════════
    elif current_state == RobotState.FETCHED:
        robot.stop()
        image_widget.value = bgr8_to_jpeg(frame)
        state_label.value  = '<h3 style="color:#ef9a9a">● State: FETCHED ✅</h3>'

        elapsed   = time.time() - fetched_at
        remaining = max(0.0, FETCHED_WAIT_SEC - elapsed)
        detail_label.value = f'<p>✅ 복귀 완료! WAITING 전환까지 {remaining:.1f}s...</p>'

        if elapsed >= FETCHED_WAIT_SEC:
            print('[STATE] FETCHED → WAITING')
            current_state      = RobotState.WAITING
            detail_label.value = ''
            zone_label.value   = ''
            servoCenter()   # 카메라 팬틸트 원점 복귀
            init_arm()      # 분류 모드 팔 위치로 복귀

print('[OK] State Machine 콜백 정의 완료.')


[OK] State Machine 콜백 정의 완료.


---
## Cell 8. ▶️ 로봇 시작

아래 셀을 실행하면 카메라 스트림에 콜백이 연결되어 자율 동작이 시작됩니다.

> **💡 Tip:** 아래 UI가 나타난 후 카메라 앞에서 손 제스처를 취해보세요.

In [66]:
# ── 상태 초기화 ───────────────────────────────────────────────────────────────
current_state   = RobotState.WAITING
fetched_at      = None
last_dance_time = 0.0
target_color    = None
current_mission = None
init_arm()  # 분류 모드 팔 위치로 초기화

# ── 카메라 콜백 연결 ──────────────────────────────────────────────────────────
camera.observe(update, names='value')

# ── UI 표시 ───────────────────────────────────────────────────────────────────
display(widgets.VBox([
    widgets.HBox([
        widgets.VBox([image_widget, zone_label]),
        widgets.VBox([info_box, prob_box]),
    ])
]))

print('[START] 로봇 시작! 손 제스처를 보여주세요 🚀')


[OK] 팔 → 분류 모드 위치 (init_arm)


[START] 로봇 시작! 손 제스처를 보여주세요 🚀


[STATE] WAITING → LINE_TRACING (class=3)
[OK] 팔 → 주행 모드 위치 (down_arm)


---
## Cell 9. ⏹ 로봇 정지 (시연 종료 시 실행)

시연이 끝나면 반드시 아래 셀을 실행하여 카메라와 로봇을 안전하게 정지시켜 주세요.

In [67]:
# ── 콜백 해제 & 로봇 / 카메라 정지 ──────────────────────────────────────────
try:
    camera.unobserve(update, names='value')
except Exception:
    pass

robot.stop()
time.sleep(0.1)
camera.stop()

print('[STOP] 로봇 정지 완료. 카메라 해제.')

[STOP] 로봇 정지 완료. 카메라 해제.


---
## 📎 부록 — 제스처 클래스 매핑 & 행동 표

| 클래스 | 손 모양 | 행동 |
|--------|---------|------|
| 0 | 주먹 ✊ | 대기 |
| **1** | **ㅗ 모양 🖕** | **라인 트레이싱 → Zone 도달 → WAITING 복귀** |
| **2** | **V자 ✌️** | **라인 트레이싱 → Zone 도달 → WAITING 복귀** |
| **3** | **손가락 3개 🤟** | **라인 트레이싱 → Zone 도달 → WAITING 복귀** |
| 4 | 손가락 4개 🖖 | 대기 |
| 5 | 손 전체 🖐️ | 대기 |

## 📎 Zone 색상 변경 방법

Cell 4의 `COLOR_UPPER` / `COLOR_LOWER` 주석을 해제하여 추적할 색상을 바꿀 수 있습니다.

여러 Zone이 필요한 경우 `check_zone()` 함수를 복수 색상 딕셔너리로 확장하세요.